In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import AIUT_Environments as environments
from math import cos, pi

# ایجاد محیط
env = environments.MountainCar()

# تست اولیه: ریست کردن محیط و چاپ وضعیت ابتدایی
initial_state = env.reset()
print(f"Initial State (Position, Velocity): {initial_state}")

# انجام یک عمل تصادفی برای تست تابع step
action = 1 # حرکت به راست
next_state, reward, terminated, truncated = env.step(action)
print(f"Next State: {next_state}, Reward: {reward}, Terminated: {terminated}")


In [ ]:
# Tile Coding parameters
num_tilings = 8
tiles_per_dim = 8

position_range = (-1.2, 0.6)
velocity_range = (-0.07, 0.07)


def tile_encode(state):
    position, velocity = state

    position_scale = tiles_per_dim / (position_range[1] - position_range[0])
    velocity_scale = tiles_per_dim / (velocity_range[1] - velocity_range[0])

    active_tiles = []

    for tiling in range(num_tilings):
        offset = tiling / num_tilings

        p = int((position - position_range[0]) * position_scale + offset)
        v = int((velocity - velocity_range[0]) * velocity_scale + offset)

        active_tiles.append((tiling, p, v))

    return active_tiles


In [ ]:
test_state = env.reset()

tiles = tile_encode(test_state)

print("Active tiles:", tiles)


In [ ]:
num_actions = 3

weights = np.zeros((num_tilings, tiles_per_dim, tiles_per_dim, num_actions))


def Q_value(state, action):
    tiles = tile_encode(state)
    value = 0

    for tiling, p, v in tiles:
        value += weights[tiling, p, v, action]

    return value


In [ ]:
epsilon = 0.1


def choose_action(state):

    if np.random.rand() < epsilon:
        return np.random.choice([0,1,2])

    values = [Q_value(state,a) for a in range(num_actions)]

    return np.argmax(values)


In [ ]:
alpha = 0.1 / num_tilings
gamma = 1.0


def update(state, action, reward, next_state, next_action):

    current = Q_value(state, action)
    target = reward + gamma * Q_value(next_state, next_action)

    delta = target - current

    tiles = tile_encode(state)

    for tiling, p, v in tiles:
        weights[tiling, p, v, action] += alpha * delta


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from AIUT_Environments import MountainCar

# ==========================================
# ۱. تنظیمات اولیه و هایپرپارامترها
# ==========================================
env = MountainCar()
num_runs = 10           # تعداد دفعات اجرای مستقل برای میانگین‌گیری
num_episodes = 5000     # تعداد اپیزودهای هر اجرا
alpha = 0.1 / 8         # نرخ یادگیری (تقسیم بر تعداد tilingها طبق کتاب)
gamma = 1.0             # ضریب تخفیف
epsilon = 0.1           # نرخ اکتشاف

# تنظیمات Tile Coding
num_tilings = 8
tiles_per_dim = 8
pos_range = [-1.2, 0.5]
vel_range = [-0.07, 0.07]

# عمل‌ها: {0: -1, 1: 0, 2: 1}
actions = [-1, 0, 1]
num_actions = 3

# محاسبه عرض کاشی‌ها
pos_tile_width = (pos_range[1] - pos_range[0]) / (tiles_per_dim - 1)
vel_tile_width = (vel_range[1] - vel_range[0]) / (tiles_per_dim - 1)

# ==========================================
# ۲. توابع کمکی (Tile Coding & Q-Value)
# ==========================================
def get_active_tiles(state):
    position, velocity = state
    active_tiles = []
    for i in range(num_tilings):
        offset = i / num_tilings
        p_idx = int(((position - pos_range[0]) / pos_tile_width) + offset)
        v_idx = int(((velocity - vel_range[0]) / vel_tile_width) + offset)
        # اطمینان از قرارگیری در بازه مجاز
        p_idx = max(0, min(p_idx, tiles_per_dim - 1))
        v_idx = max(0, min(v_idx, tiles_per_dim - 1))
        active_tiles.append((i, p_idx, v_idx))
    return active_tiles

def get_q_value(state, action_idx, weights):
    tiles = get_active_tiles(state)
    q_val = 0
    for t in tiles:
        q_val += weights[t[0], t[1], t[2], action_idx]
    return q_val

def choose_action(state, weights):
    if np.random.random() < epsilon:
        return np.random.choice(num_actions)
    else:
        q_values = [get_q_value(state, a, weights) for a in range(num_actions)]
        return np.argmax(q_values)

# ==========================================
# ۳. حلقه اصلی آموزش (10 Runs)
# ==========================================
all_steps = np.zeros((num_runs, num_episodes))

print("شروع فرآیند آموزش (این مرحله ممکن است چند دقیقه طول بکشد)...")

for run in range(num_runs):
    # مقداردهی اولیه وزن‌ها (تنسور ۴ بعدی)
    weights = np.zeros((num_tilings, tiles_per_dim, tiles_per_dim, num_actions))

    for ep in range(num_episodes):
        state = env.reset()
        action_idx = choose_action(state, weights)
        steps = 0

        while True:
            next_state, reward, terminated, truncated = env.step(actions[action_idx])

            # محاسبه خطای TD و آپدیت وزن‌ها (Semi-gradient SARSA)
            current_q = get_q_value(state, action_idx, weights)

            if terminated or truncated:
                target = reward
            else:
                next_action_idx = choose_action(next_state, weights)
                next_q = get_q_value(next_state, next_action_idx, weights)
                target = reward + gamma * next_q

            # آپدیت وزن‌های کاشی‌های فعال
            td_error = target - current_q
            tiles = get_active_tiles(state)
            for t in tiles:
                weights[t[0], t[1], t[2], action_idx] += alpha * td_error

            if terminated or truncated:
                break

            state = next_state
            action_idx = next_action_idx
            steps += 1

        all_steps[run, ep] = steps

    print(f"اجرای مستقل {run + 1}/{num_runs} به پایان رسید.")

# محاسبه میانگین
avg_steps = np.mean(all_steps, axis=0)

# ==========================================
# ۴. رسم نمودار نهایی
# ==========================================
plt.figure(figsize=(11, 6))
plt.plot(avg_steps, color='#2c3e50', linewidth=1, label='Average Steps')
# اضافه کردن یک میانگین متحرک برای زیبایی و تحلیل بهتر (اختیاری)
moving_avg = np.convolve(avg_steps, np.ones(50)/50, mode='valid')
plt.plot(range(50, 5001), moving_avg, color='#e74c3c', linewidth=2, label='Moving Average (50 ep)')

plt.title("Learning Curve: Semi-gradient SARSA with Tile Coding\n(Average of 10 Independent Runs)", fontsize=14)
plt.xlabel("Episodes", fontsize=12)
plt.ylabel("Steps per Episode", fontsize=12)
plt.yscale('log') # استفاده از مقیاس لگاریتمی برای نمایش بهتر افت اولیه
plt.grid(True, which="both", ls="-", alpha=0.5)
plt.legend()
plt.show()


In [ ]:
from matplotlib import cm

def plot_cost_to_go(weights, num_tilings, tiles_per_dim):
    # ایجاد شبکه نقاط برای موقعیت و سرعت
    positions = np.linspace(pos_range[0], pos_range[1], 40)
    velocities = np.linspace(vel_range[0], vel_range[1], 40)
    px, vx = np.meshgrid(positions, velocities)

    # محاسبه Cost-to-go (منفیِ بیشترین مقدار Q)
    cost_to_go = np.zeros(px.shape)

    for i in range(len(positions)):
        for j in range(len(velocities)):
            state = [positions[i], velocities[j]]
            # محاسبه V(s) = max_a Q(s, a)
            q_values = [get_q_value(state, a, weights) for a in range(num_actions)]
            cost_to_go[j, i] = -np.max(q_values) # Cost = -V(s)

    # رسم نمودار سه بعدی
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(px, vx, cost_to_go, cmap=cm.terrain,
                           linewidth=0.1, antialiased=True)

    ax.set_title("Cost-to-go Function (Final Episode)", fontsize=15)
    ax.set_xlabel('Position', fontsize=12)
    ax.set_ylabel('Velocity', fontsize=12)
    ax.set_zlabel('Cost-to-go', fontsize=12)

    # تنظیم زاویه دید مشابه کتاب ساتن
    ax.view_init(elev=30, azim=150)

    fig.colorbar(surf, shrink=0.5, aspect=10)
    plt.show()

# فراخوانی تابع
plot_cost_to_go(weights, num_tilings, tiles_per_dim)
